# 🌊 Flood Risk Final Submission
.

## 📥 Step 1: Install & Import Libraries

In [1]:
!pip install catboost xgboost lightgbm scikit-learn pandas numpy -q

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.cluster import KMeans
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder, TargetEncoder
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")
print('Libraries imported')

Libraries imported


## ⚙️ Step 2: Configuration & Auto-Path Detection

In [2]:
def get_data_path(filename):
    kaggle_dir = "/kaggle/input/"
    if os.path.exists(kaggle_dir):
        for root, dirs, files in os.walk(kaggle_dir):
            if filename in files:
                return os.path.join(root, filename)
    return filename # Fallback to local directory

class Config:
    TARGET_COLUMN = "flood_risk_score"
    ID_COLUMN = "record_id"
    COLUMNS_TO_DROP = ["place_name", "generation_date", "is_synthetic", "reason_not_good_to_live"]
    CATEGORICAL_COLUMNS = [
        "district", "landcover", "soil_type", "water_supply", "electricity",
        "road_quality", "urban_rural", "water_presence_flag",
        "flood_occurrence_current_event", "is_good_to_live",
        "geo_cluster" # Added dynamically
    ]
    TARGET_ENCODE_COLUMNS = ["place_name", "geo_cluster", "district"]
    SUBMISSION_FILE = "submission.csv" # Standard Kaggle submission name
    
    TRAIN_FILE = get_data_path("train.csv")
    TEST_FILE = get_data_path("test.csv")
    SAMPLE_SUBMISSION_FILE = get_data_path("sample_submission.csv")
    
    # --- HEAVY TRAINING PARAMS (Anti-Overfitting) ---
    LEARNING_RATE = 0.008
    MAX_ITER = 2500
    MAX_DEPTH = 8
    MIN_SAMPLES_LEAF = 50
    L2_REG = 2.0
    RANDOM_STATE = 42
    TEST_SIZE = 0.2
    
print(f"Train path detected as: {Config.TRAIN_FILE} ")

Train path detected as: /kaggle/input/competitions/ml-opsidian-genesis-initial-round-26/train.csv 


## 📊 Step 3: Load Data & Leak Check #1

In [3]:
train_df = pd.read_csv(Config.TRAIN_FILE)
test_df = pd.read_csv(Config.TEST_FILE)
sample_sub = pd.read_csv(Config.SAMPLE_SUBMISSION_FILE)

ORIGINAL_TRAIN_LEN = len(train_df)
ORIGINAL_TEST_LEN = len(test_df)
ORIGINAL_SUB_LEN = len(sample_sub)
print(f"Loaded Data:\nTrain: {ORIGINAL_TRAIN_LEN} rows\nTest: {ORIGINAL_TEST_LEN} rows")

# --- LEAK CHECK 1 ---
assert ORIGINAL_TEST_LEN == ORIGINAL_SUB_LEN, "CRITICAL ERROR: Test data and Submission lengths don't match!"
print("Leak Check 1 Passed: Row counts are 100% correct.")

Loaded Data:
Train: 20886 rows
Test: 5300 rows
Leak Check 1 Passed: Row counts are 100% correct.


## 🧹 Step 4: Perfect Data Cleaning & Leak Check #2

In [4]:
cols_to_drop = [col for col in Config.COLUMNS_TO_DROP if col in train_df.columns]
train_df = train_df.drop(columns=cols_to_drop)
test_df = test_df.drop(columns=cols_to_drop, errors="ignore")

train_df["missing_count"] = train_df.isnull().sum(axis=1)
test_df["missing_count"] = test_df.isnull().sum(axis=1)

exclude = {Config.ID_COLUMN, Config.TARGET_COLUMN}

for col in train_df.columns:
    if col in exclude: continue
    has_train_missing = train_df[col].isnull().sum() > 0
    has_test_missing = col in test_df.columns and test_df[col].isnull().sum() > 0
    if has_train_missing or has_test_missing:
        train_df[f"{col}_is_missing"] = train_df[col].isnull().astype(int)
        if col in test_df.columns:
            test_df[f"{col}_is_missing"] = test_df[col].isnull().astype(int)
        
        if train_df[col].dtype in ["float64", "int64", "float32", "int32"]:
            fill_value = train_df[col].median()
        else:
            fill_value = train_df[col].mode()[0] if not train_df[col].mode().empty else "Unknown"
        
        train_df[col] = train_df[col].fillna(fill_value)
        if col in test_df.columns:
            test_df[col] = test_df[col].fillna(fill_value)

# --- LEAK CHECK 2 ---
assert len(train_df) == ORIGINAL_TRAIN_LEN, "CRITICAL ERROR: Train rows dropped!"
assert len(test_df) == ORIGINAL_TEST_LEN, "CRITICAL ERROR: Test rows dropped!"
print("Leak Check 2 Passed: Data Cleaning was 100% strictly isolated.")

Leak Check 2 Passed: Data Cleaning was 100% strictly isolated.


## 🌍 Step 5: Leak-Free Spatial Features (KMeans) & Leak Check #3

In [5]:
if "latitude" in train_df.columns and "longitude" in train_df.columns:
    print("Creating Geo Clusters...")
    kmeans = KMeans(n_clusters=35, random_state=Config.RANDOM_STATE, n_init=10)
    kmeans.fit(train_df[["latitude", "longitude"]])
    train_df["geo_cluster"] = kmeans.predict(train_df[["latitude", "longitude"]]).astype(str)
    test_df["geo_cluster"] = kmeans.predict(test_df[["latitude", "longitude"]]).astype(str)

# --- LEAK CHECK 3 ---
assert "flood_risk_score" not in kmeans.feature_names_in_, "CRITICAL ERROR: KMeans cheated by looking at the target!"
print("Leak Check 3 Passed: Spatial features generated purely from coordinates.")

Creating Geo Clusters...
Leak Check 3 Passed: Spatial features generated purely from coordinates.


## 🧬 Step 6: Engineering & Leak Check #4

In [6]:
y_train = train_df[Config.TARGET_COLUMN].copy()

categorical_cols = [col for col in Config.CATEGORICAL_COLUMNS if col in train_df.columns]
if categorical_cols:
    for col in categorical_cols:
        le = LabelEncoder()
        all_values = pd.concat([train_df[col].astype(str), test_df[col].astype(str)], ignore_index=True)
        le.fit(all_values)
        train_df[col] = le.transform(train_df[col].astype(str))
        test_df[col] = le.transform(test_df[col].astype(str))

target_encode_cols = [col for col in Config.TARGET_ENCODE_COLUMNS if col in train_df.columns]
if target_encode_cols:
    print("Applying OOF Target Encoding (Internal Cross-Validation prevents leaks)...")
    te = TargetEncoder(random_state=Config.RANDOM_STATE, cv=5)
    train_df[target_encode_cols] = te.fit_transform(train_df[target_encode_cols], y_train)
    test_df[target_encode_cols] = te.transform(test_df[target_encode_cols])

for df in [train_df, test_df]:
    if "rainfall_7d_mm" in df.columns and "monthly_rainfall_mm" in df.columns:
        df["rainfall_intensity_ratio"] = df["rainfall_7d_mm"] / (df["monthly_rainfall_mm"] + 1)
    if "nearest_hospital_km" in df.columns and "nearest_evac_km" in df.columns:
        df["accessibility_score"] = df["nearest_hospital_km"] + df["nearest_evac_km"]
    if "population_density_per_km2" in df.columns and "infrastructure_score" in df.columns:
        df["population_vulnerability"] = df["population_density_per_km2"] / (df["infrastructure_score"] + 1)
    if "elevation_m" in df.columns and "distance_to_river_m" in df.columns:
        df["elevation_river_ratio"] = df["elevation_m"] / (df["distance_to_river_m"] + 1)
        df["rain_vs_river"] = df.get("rainfall_7d_mm", 0) / (df["distance_to_river_m"] + 1)
        df["extreme_risk_index"] = df["rain_vs_river"] * (1 / (df["elevation_m"] + 1))

test_ids = test_df[Config.ID_COLUMN].copy()
cols_to_exclude = {Config.TARGET_COLUMN, Config.ID_COLUMN}
feature_cols = [col for col in train_df.columns if col not in cols_to_exclude and col in test_df.columns]

X_train = train_df[feature_cols].copy().fillna(0)
X_test = test_df[feature_cols].copy().fillna(0)

non_numeric = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    X_train = X_train.drop(columns=non_numeric)
    X_test = X_test.drop(columns=non_numeric)

# --- LEAK CHECK 4 ---
assert len(X_train) == ORIGINAL_TRAIN_LEN, "CRITICAL ERROR: Lost rows in X_train!"
assert len(X_test) == ORIGINAL_TEST_LEN, "CRITICAL ERROR: Lost rows in X_test!"
print("Leak Check 4 Passed: Feature Engineering complete without data loss.")

Applying OOF Target Encoding (Internal Cross-Validation prevents leaks)...
Leak Check 4 Passed: Feature Engineering complete without data loss.


## 🤖 Step 7: The Stacking Meta-Ensemble

In [7]:
print("Building the Ultimate Stacking Regressor...")

model_hgb = HistGradientBoostingRegressor(
    max_iter=Config.MAX_ITER, learning_rate=Config.LEARNING_RATE, max_depth=Config.MAX_DEPTH,
    min_samples_leaf=Config.MIN_SAMPLES_LEAF, l2_regularization=Config.L2_REG, random_state=Config.RANDOM_STATE, early_stopping=True
)

model_xgb = xgb.XGBRegressor(
    n_estimators=2000, learning_rate=Config.LEARNING_RATE, max_depth=Config.MAX_DEPTH, 
    subsample=0.8, colsample_bytree=0.8, reg_lambda=Config.L2_REG, 
    random_state=Config.RANDOM_STATE, tree_method='hist'
)

model_lgb = lgb.LGBMRegressor(
    n_estimators=2500, learning_rate=Config.LEARNING_RATE, max_depth=Config.MAX_DEPTH,
    num_leaves=45, subsample=0.8, colsample_bytree=0.8, min_child_samples=Config.MIN_SAMPLES_LEAF,
    reg_lambda=Config.L2_REG, random_state=Config.RANDOM_STATE, n_jobs=-1, verbose=-1
)

model_cat = CatBoostRegressor(
    iterations=2500, learning_rate=Config.LEARNING_RATE, depth=Config.MAX_DEPTH,
    l2_leaf_reg=Config.L2_REG, random_seed=Config.RANDOM_STATE, verbose=False
)

stacker = StackingRegressor(
    estimators=[
        ('hgb', model_hgb),
        ('xgb', model_xgb),
        ('lgb', model_lgb),
        ('cat', model_cat)
    ],
    final_estimator=RidgeCV(),
    cv=5,
    n_jobs=-1
)

print("\nTraining on Full Dataset (This will take a few minutes)...\n(Cross Validation is handled natively by Stacker)")
stacker.fit(X_train, y_train)
print("\nTraining Complete!")

Building the Ultimate Stacking Regressor...

Training on Full Dataset (This will take a few minutes)...
(Cross Validation is handled natively by Stacker)

Training Complete!


## 💾 Step 8: Final Checks & Kaggle Submission

In [8]:
predictions = np.clip(stacker.predict(X_test), 0.0, 1.0)

submission = pd.DataFrame({
    sample_sub.columns[0]: test_ids,
    sample_sub.columns[1]: np.round(predictions, 5),
})

# --- LEAK CHECK 5 ---
assert len(submission) == ORIGINAL_SUB_LEN, "CRITICAL ERROR: Submission row count does not match Kaggle requirements!"
assert submission[sample_sub.columns[1]].isnull().sum() == 0, "CRITICAL ERROR: Nulls in predictions!"
print("Leak Check 5 Passed: Final submission is mathematically perfectly shaped.")

submission.to_csv(Config.SUBMISSION_FILE, index=False)
print(f"\nSuccessfully created {Config.SUBMISSION_FILE} \nReady to submit!")
import joblib
joblib.dump(stacker, 'flood_risk_model.pkl')
print('\nModel successfully saved as flood_risk_model.pkl')


Leak Check 5 Passed: Final submission is mathematically perfectly shaped.

Successfully created submission.csv 
Ready to submit!
